# Merge Sydney station files (2019–2024)

This notebook merges the yearly NSW air-quality `.xls` files into one file per station:

- ROZELLE_2019_2024.csv
- RANDWICK_2019_2024.csv
- LIVERPOOL_2019_2024.csv
- ST_MARYS_2019_2024.csv
- BRINGELLY_2019_2024.csv
- RICHMOND_2019_2024.csv

## Before you run
1. Create a folder called `data/raw`
2. Put all your downloaded `.xls` files inside that folder
3. Run the cells from top to bottom

The notebook will save the merged outputs into `data/processed`.


## 1) Imports

Run this first.


In [1]:
import pandas as pd
from pathlib import Path
import re
import warnings

import sys
!{sys.executable} -m pip install xlrd

warnings.filterwarnings("ignore")

raw_dir = Path("data/raw")
out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

print("Raw folder:", raw_dir.resolve())
print("Processed folder:", out_dir.resolve())
print("Raw exists?:", raw_dir.exists())

c:\Users\MSI\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\MSI\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Raw folder: C:\Users\MSI\Desktop\Research Perp\Code\data\raw
Processed folder: C:\Users\MSI\Desktop\Research Perp\Code\data\processed
Raw exists?: True


## 2) Helper functions

These functions:
- clean column names
- detect the station name
- read one `.xls` file
- standardise the datetime column


In [2]:
STATIONS = [
    "ROZELLE",
    "RANDWICK",
    "LIVERPOOL",
    "ST_MARYS",
    "BRINGELLY",
    "RICHMOND",
]

def clean_col(name):
    name = str(name).strip()
    name = re.sub(r"\s+", "_", name)
    name = name.replace("/", "_")
    name = name.replace("(", "").replace(")", "")
    name = name.replace("[", "").replace("]", "")
    name = name.replace("°", "")
    name = name.replace(".", "")
    return name.upper()

def normalise_station_label(text):
    text = str(text).upper().strip()
    text = text.replace("ST MARYS", "ST_MARYS")
    text = text.replace("ST._MARYS", "ST_MARYS")
    text = text.replace("ST-MARYS", "ST_MARYS")
    return text

def detect_station_from_text(text):
    text = normalise_station_label(text)
    for st in STATIONS:
        if st in text:
            return st
    return None

def detect_station_from_columns(columns):
    for c in columns:
        st = detect_station_from_text(c)
        if st:
            return st
    return None

def detect_datetime_col(columns):
    for c in columns:
        c_upper = str(c).upper()
        if "DATE" in c_upper or "TIME" in c_upper:
            return c
    return None

def read_one_file(fp):
    last_error = None

    for header_row in [0, 1, 2, 3, 4]:
        try:
            df = pd.read_excel(fp, header=header_row, engine="xlrd")
            df.columns = [clean_col(c) for c in df.columns]

            # detect separate date and time columns
            date_col = None
            time_col = None

            for c in df.columns:
                c_upper = str(c).upper()
                if c_upper == "DATE" or c_upper.startswith("DATE_"):
                    date_col = c
                if c_upper == "TIME" or c_upper.startswith("TIME_"):
                    time_col = c

            # case 1: separate DATE and TIME columns
            if date_col is not None and time_col is not None:
                df["DATETIME"] = pd.to_datetime(
                    df[date_col].astype(str).str.strip() + " " + df[time_col].astype(str).str.strip(),
                    errors="coerce",
                    dayfirst=True
                )

            # case 2: already combined datetime-like column
            else:
                dt_col = detect_datetime_col(df.columns)
                if dt_col is None:
                    last_error = f"No datetime column with header={header_row}"
                    continue

                df["DATETIME"] = pd.to_datetime(df[dt_col], errors="coerce", dayfirst=True)

            df = df.dropna(subset=["DATETIME"]).copy()

            station = detect_station_from_columns(df.columns)
            if station is None:
                station = detect_station_from_text(fp.stem)

            if station is None:
                raise ValueError(f"Could not detect station name from {fp.name}")

            df["STATION"] = station
            df = df.sort_values("DATETIME").drop_duplicates(subset=["DATETIME"]).reset_index(drop=True)

            print(f"{fp.name}: success with header={header_row}")
            return df, station

        except Exception as e:
            last_error = str(e)

    raise ValueError(f"{fp.name}: failed after trying multiple header rows. Last error: {last_error}")

## 3) Check your input files

This shows how many `.xls` files the notebook can see.
You should have your station-year files inside `data/raw`.


In [3]:
files = sorted(raw_dir.glob("*.xls"))
print("Number of .xls files found:", len(files))
for f in files[:10]:
    print("-", f.name)

if len(files) == 0:
    print("\nNo .xls files found. Put your files inside data/raw first.")

Number of .xls files found: 36
- tmp_table_11341_1782984867.xls
- tmp_table_14586_1782986049.xls
- tmp_table_15134_1782983864.xls
- tmp_table_16139_1782985301.xls
- tmp_table_16212_1782985136.xls
- tmp_table_16759_1782985784.xls
- tmp_table_20440_1782983651.xls
- tmp_table_22008_1782985948.xls
- tmp_table_22234_1782983713.xls
- tmp_table_22253_1782985373.xls


## 4) Read all files and group by station

This loads each file and groups it under its station.


In [4]:
station_frames = {}
load_log = []

for fp in files:
    try:
        df, station = read_one_file(fp)
        station_frames.setdefault(station, []).append(df)
        load_log.append((fp.name, station, len(df), str(df["DATETIME"].min()), str(df["DATETIME"].max())))
        print(f"Loaded {fp.name} -> {station} | rows={len(df)}")
    except Exception as e:
        print(f"FAILED {fp.name}: {e}")

print("\nStations found:", list(station_frames.keys()))

WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
tmp_table_11341_1782984867.xls: success with header=2
Loaded tmp_table_11341_1782984867.xls -> LIVERPOOL | rows=8418
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
tmp_table_14586_1782986049.xls: success with header=2
Loaded tmp_table_14586_1782986049.xls -> RICHMOND | rows=8395
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
tmp_table_15134_1782983864.xls: success with header=2
Loaded tmp_table_15134_1782983864.xls -> RANDWICK | rows

## 5) Merge yearly files into one file per station

This is the main step.
It stacks all years for each station and saves one merged CSV file per station.


In [5]:
merged_station_files = {}

for station, dfs in station_frames.items():
    merged = pd.concat(dfs, ignore_index=True)
    merged = merged.sort_values("DATETIME")
    merged = merged.drop_duplicates(subset=["DATETIME"]).reset_index(drop=True)
    merged["YEAR"] = merged["DATETIME"].dt.year

    out_path = out_dir / f"{station}_2019_2024.csv"
    merged.to_csv(out_path, index=False)

    merged_station_files[station] = merged

    years = sorted(merged["YEAR"].dropna().unique().tolist())
    print(f"Saved {out_path.name} | rows={len(merged)} | years={years}")

Saved LIVERPOOL_2019_2024.csv | rows=50416 | years=[2019, 2020, 2021, 2022, 2023, 2024]
Saved RICHMOND_2019_2024.csv | rows=50416 | years=[2019, 2020, 2021, 2022, 2023, 2024]
Saved RANDWICK_2019_2024.csv | rows=50416 | years=[2019, 2020, 2021, 2022, 2023, 2024]
Saved ST_MARYS_2019_2024.csv | rows=50416 | years=[2019, 2020, 2021, 2022, 2023, 2024]
Saved BRINGELLY_2019_2024.csv | rows=50416 | years=[2019, 2020, 2021, 2022, 2023, 2024]
Saved ROZELLE_2019_2024.csv | rows=50416 | years=[2019, 2020, 2021, 2022, 2023, 2024]


## 6) Quick quality check

This helps you verify:
- start and end date
- duplicate timestamps
- top missing columns


In [6]:
for station, df in merged_station_files.items():
    print(f"\n--- {station} ---")
    print("Rows:", len(df))
    print("Start:", df["DATETIME"].min())
    print("End:", df["DATETIME"].max())
    print("Duplicate DATETIME:", df["DATETIME"].duplicated().sum())
    print("Top missing columns:")
    print(df.isna().sum().sort_values(ascending=False).head(12))


--- LIVERPOOL ---
Rows: 50416
Start: 2019-01-01 01:00:00
End: 2024-12-31 23:00:00
Duplicate DATETIME: 0
Top missing columns:
LIVERPOOL_SOLAR_1H_AVERAGE_W_M²    50416
LIVERPOOL_RAIN_1H_AVERAGE_MM_M²    22682
LIVERPOOL_CO_1H_AVERAGE_PPM         4785
LIVERPOOL_NO2_1H_AVERAGE_PPHM       3584
LIVERPOOL_OZONE_1H_AVERAGE_PPHM     3415
LIVERPOOL_PM25_1H_AVERAGE_ΜG_M³     1406
LIVERPOOL_PM10_1H_AVERAGE_ΜG_M³      771
LIVERPOOL_HUMID_1H_AVERAGE_%         710
LIVERPOOL_TEMP_1H_AVERAGE_C          705
LIVERPOOL_WDR_1H_AVERAGE_            333
LIVERPOOL_WSP_1H_AVERAGE_M_S         333
DATE                                   0
dtype: int64

--- RICHMOND ---
Rows: 50416
Start: 2019-01-01 01:00:00
End: 2024-12-31 23:00:00
Duplicate DATETIME: 0
Top missing columns:
RICHMOND_CO_1H_AVERAGE_PPM        50416
RICHMOND_SOLAR_1H_AVERAGE_W_M²    50416
RICHMOND_PM25_1H_AVERAGE_ΜG_M³     4723
RICHMOND_NO2_1H_AVERAGE_PPHM       4186
RICHMOND_OZONE_1H_AVERAGE_PPHM     4155
RICHMOND_PM10_1H_AVERAGE_ΜG_M³     2031
RICH

## 7) Save a merge summary table

This creates a small CSV summary you can inspect later.


In [7]:
summary_rows = []

for station, df in merged_station_files.items():
    summary_rows.append({
        "station": station,
        "rows": len(df),
        "start": df["DATETIME"].min(),
        "end": df["DATETIME"].max(),
        "duplicate_datetime": int(df["DATETIME"].duplicated().sum()),
        "n_columns": df.shape[1],
    })

summary_df = pd.DataFrame(summary_rows).sort_values("station")
summary_df.to_csv(out_dir / "station_merge_summary.csv", index=False)
summary_df

,station,rows,start,end,duplicate_datetime,n_columns
4,BRINGELLY,50416,2019-01-01 01:00:00,2024-12-31 23:00:00,0,15
0,LIVERPOOL,50416,2019-01-01 01:00:00,2024-12-31 23:00:00,0,16
2,RANDWICK,50416,2019-01-01 01:00:00,2024-12-31 23:00:00,0,14
1,RICHMOND,50416,2019-01-01 01:00:00,2024-12-31 23:00:00,0,16
5,ROZELLE,50416,2019-01-01 01:00:00,2024-12-31 23:00:00,0,16
3,ST_MARYS,50416,2019-01-01 01:00:00,2024-12-31 23:00:00,0,15


## What to do after this

After this notebook works, your next step is:

1. Check each merged station CSV
2. Confirm missing variables
3. Keep the 6 station files separate for now
4. Later build:
   - Rozelle = target/local
   - 5 neighbour station datasets
   - context dataset (bushfire / traffic)
   - final aligned modelling table
